# Expected Value Engine

Score every pitch in the 2025 modeling frame with the **RF probability chain** and compute **expected run value (ERV)** for the pitcher.

**Prerequisites:** `03_feature_engineering.ipynb` → `05_modeling_swing_rf.ipynb` → `07_modeling_whiff_rf.ipynb` → `09_modeling_xwobacon_rf.ipynb`

**Input:** `data/modeling_frame_2025.parquet`

**Output:** `data/ev_scored_pitches.parquet`

Logic lives in `src/ev_calculator.py`.

## Model chain architecture

| Model | Artifact | Training sample | Feature selection |
|-------|----------|-----------------|-------------------|
| **A — Swing** | `swing_rf_master.joblib` | All pitches | Forward selection (Aug val log-loss, MIN_GAIN = 0.0010) |
| **B — Whiff** | `whiff_rf_master.joblib` | Swings only | Forward selection (same threshold) |
| **C — xwOBAcon** | `xwobacon_rf_master.joblib` | BIP only | Kitchen-sink (all groups combined) |

**Baseline (all three):** `is_in_zone`, `miss_dist_in`, `center_dist_in`

**Batter signal:** season-to-date rolling rates in the modeling frame (`swing_pct`, `whiff_pct`, zone splits). **No `batter_te`.** Whiff and xwOBAcon also use rolling bat-tracking (`bat_speed`, `attack_angle`, `squared_up_rate`).

**Swing group order:** `pitch_type` → `pitch_quality` → `count` → `batter_rolling` → `pitch_movement` → `pitcher_delivery` → `attack_zone` → `base_state` → `platoon`

**Whiff / xwOBAcon group order:** `pitch_type` → `pitch_quality` → `count` → `batter_rolling` → `bat_tracking` → `pitch_movement` → `pitcher_delivery` → `attack_zone` → `base_state` → `platoon`


## 1. ERV formula

| Step | Choice |
|------|--------|
| **Model A** | `p_swing` from swing RF |
| **Model B** | `p_whiff_given_swing` from whiff RF (trained on swings only) |
| **Model C** | `pred_xwobacon` from xwOBAcon RF (trained on BIP only) |
| **Inference features** | Pre-engineered columns from `03` — rolling batter profile, not target-encoded IDs |
| **Take value** | `RV_STRIKE = -0.065` if `is_in_zone == 1`, else `RV_BALL = +0.035` |
| **Contact value** | `(pred_xwobacon - 0.320) / 1.15` |
| **Swing value** | `p_whiff × RV_WHIFF + p_contact × contact_rv` (`RV_WHIFF = -0.095`) |
| **ERV** | `p_take × take_value + p_swing × swing_value` |

Lower (more negative) ERV is better for the pitcher.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

NB_DIR = Path.cwd().resolve()
ROOT = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import importlib
import src.ev_calculator as ev_calculator

importlib.reload(ev_calculator)

from src.ev_calculator import (
    DEFAULT_INPUT,
    DEFAULT_OUTPUT,
    LEAGUE_WOBA,
    RV_BALL,
    RV_STRIKE,
    RV_WHIFF,
    WOBA_SCALE,
    calculate_expected_run_value,
    describe_ev_artifacts,
    load_ev_artifacts,
    print_verification_examples,
)
from src.feature_engineering import BASELINE_FEATURES

MODELING_FILE = ROOT / "data" / "modeling_frame_2025.parquet"
if not MODELING_FILE.exists():
    raise FileNotFoundError(
        f"Run 03_feature_engineering.ipynb first — missing {MODELING_FILE}"
    )

artifacts = load_ev_artifacts(ROOT / "models")
print("Baseline features (all models):", BASELINE_FEATURES)
print()
display(describe_ev_artifacts(artifacts))


## 2. Score all pitches

In [ ]:
pitches = pd.read_parquet(MODELING_FILE)
print(f"Input rows: {len(pitches):,}")

scored = calculate_expected_run_value(pitches, artifacts=artifacts)

new_cols = [
    "p_swing",
    "p_take",
    "p_whiff_given_swing",
    "p_contact_given_swing",
    "pred_xwobacon",
    "expected_run_value",
]
print(f"Added columns: {new_cols}")
print(f"\nERV summary (pitcher perspective — lower is better):")
print(scored["expected_run_value"].describe().round(4))


In [ ]:
DEFAULT_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
scored.to_parquet(DEFAULT_OUTPUT, index=False)
print(f"Saved -> {DEFAULT_OUTPUT}")


## 3. ERV distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(scored["expected_run_value"], bins=80, color="#6366f1", edgecolor="white", linewidth=0.3)
ax.axvline(0, color="gray", linestyle="--", linewidth=1)
ax.set_title("Expected run value — all 2025 pitches")
ax.set_xlabel("ERV (runs; lower = better for pitcher)")
ax.set_ylabel("Pitch count")
fig.tight_layout()
plt.show()


## 4. Verification examples

Three hand-picked pitches with the full ERV math printed step-by-step.

In [ ]:
print_verification_examples(scored)

## 5. Summary

The EV engine chains three RF models into a single **expected_run_value** per pitch.

- **Artifacts:** `swing_rf_master.joblib`, `whiff_rf_master.joblib`, `xwobacon_rf_master.joblib`
- **Features:** baseline location + forward-selected / kitchen-sink groups with rolling batter metrics
- **Output parquet:** `data/ev_scored_pitches.parquet` with probability columns + ERV
- **Next:** `11_ai_catcher.ipynb` — grid-search pitch recommendations using this chain